In [ ]:
import pandas as pd

In [ ]:
# These names come from the NASA documentation
index_names = ['unit_nr', 'time_cycles']
setting_names = ['setting_1', 'setting_2', 'setting_3']
sensor_names = [f's_{i}' for i in range(1, 22)] 
col_names = index_names + setting_names + sensor_names

In [ ]:
# Path to your file based on your listdir result
train_path = '../data/train_FD001.txt'

# Load it, header is set to None as they are manually defined
df_train = pd.read_csv(train_path, sep='\s+', header=None, names=col_names)

# Look at the "Shape" and the first few rows. We use 0 to count number
# of rows and 1 to count number of columns to verify the data
print(f"Rows: {df_train.shape[0]}, Columns: {df_train.shape[1]}")

In [ ]:
#Transpose the table so variables show as rows instead of columns
stats = df_train.describe().T

# Calculate how much each column varies
# A standard deviation of 0 means the sensor never changes
stats['std']

In [ ]:
# We identify the columns that show a standard deviation of 0
# to reduce dimensionality and prevent
dead_columns = stats[stats['std'] == 0].index.tolist()

print(f"Dropping these useless columns: {dead_columns}")

# 2. We drop them from our main DataFrame
df_train_cleaned = df_train.drop(columns=dead_columns)

# 3. Check the new shape
print(f"New Shape: {df_train_cleaned.shape}")

In [ ]:
# To find the maximum number of cycles for each engine we group by
# unit number and find the max value in time cycles, creating a list.
# We reset the index to make the label into a column so it can be manipulated
max_cycles = df_train_cleaned.groupby('unit_nr')['time_cycles'].max().reset_index()
max_cycles.columns = ['unit_nr', 'max_life']

max_cycles.head()

In [ ]:
# We left join the max cycles table onto the train table using the unit number
df_train_cleaned = df_train_cleaned.merge(max_cycles, on='unit_nr', how='left')

# Preview the merged table
# Now look at the first few rows to see the new column at the end
df_train_cleaned.head()

In [ ]:
# We create the remaining useful life (RUL) column to show how
# many cycles are left. RUL = (max life) - (time cycles)
df_train_cleaned['RUL'] = df_train_cleaned['max_life'] - df_train_cleaned['time_cycles']

# Preview only the required columns to verify
df_train_cleaned[['unit_nr', 'time_cycles', 'max_life', 'RUL']].head()

In [ ]:
# Check the last rows of unit 1 to see the RUL reaches 0
df_train_cleaned[df_train_cleaned['unit_nr'] == 1].tail()

In [ ]:
# Now RUL is calculated we drop the max life column
df_train_cleaned = df_train_cleaned.drop(columns=['max_life'])

# Preview the cleaned table
df_train_cleaned.head()

In [ ]:
from sklearn.preprocessing import MinMaxScaler

# 1. Initialize the Scaler
scaler = MinMaxScaler()

# Identify the columns to scale, we don't scale unit number, time cycles
# or remaining useful life, using difference to remove them from the list
cols_to_scale = df_train_cleaned.columns.difference(['unit_nr', 'time_cycles', 'RUL'])

# Fit and transform the remaining columns so they are scaled
df_train_cleaned[cols_to_scale] = scaler.fit_transform(df_train_cleaned[cols_to_scale])

# Preview the dataframe
df_train_cleaned.head()

In [ ]:
# Look at the stats of your scaled sensors to identify redundant features
df_train_cleaned[cols_to_scale].describe().T[['min', 'max', 'mean', 'std']]

In [ ]:
# Count columns that have more than one unique value
real_sensors = df_train_cleaned.columns[df_train_cleaned.nunique() > 1].tolist()

#Identify the names and number of useful columns
print(f"Number of useful columns: {len(real_sensors)}")
print(f"useful columns: {real_sensors}")

In [ ]:
# Update the dataframe to include only useful features
df_train_final = df_train_cleaned[real_sensors]

df_train_final.head()


In [ ]:
# We now define X and y using the finalized dataframe
# We drop unit number because it is an ID and not a sensor
# We drop remaining useful life because it is the target
X = df_train_final.drop(columns=['unit_nr', 'RUL'])
y = df_train_final['RUL']

# Preview the final features
print(f"Final features: {X.columns.tolist()}")

In [ ]:
# To build the new feature set, we additionally remove time cycles
X_honest = df_train_final.drop(columns=['unit_nr', 'RUL', 'time_cycles'])


In [ ]:
# Create the clipped labels
y_train_clipped = y_train_h.clip(upper=125)
y_test_clipped = y_test_h.clip(upper=125)

In [ ]:
# Update the dataframe and make it an independent copy
df_train_final = df_train_cleaned[real_sensors].copy()

# Defining the top 4 importances, we calculate the rolling mean
# and standard deviation using a window size of 10
for col in top_4:
    df_train_final[f'{col}_roll_mean'] = df_train_final.groupby('unit_nr')[col].transform(
        lambda x: x.rolling(window=10, min_periods=1).mean())
    df_train_final[f'{col}_roll_std'] = df_train_final.groupby('unit_nr')[col].transform(
        lambda x: x.rolling(window=10, min_periods=1).std().fillna(0))

In [2]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler

# 1. SETUP CONSTANTS
INDEX_NAMES = ['unit_nr', 'time_cycles']
SETTING_NAMES = ['setting_1', 'setting_2', 'setting_3']
SENSOR_NAMES = [f's_{i}' for i in range(1, 22)] 
COL_NAMES = INDEX_NAMES + SETTING_NAMES + SENSOR_NAMES

# The sensors we keep (excluding 'Zombie' sensors and non-informative settings)
# This replaces the need for 'real_sensors' logic later
FEAT_COLS = [
    'setting_1', 's_2', 's_3', 's_4', 's_7', 's_8', 's_9', 's_11', 
    's_12', 's_13', 's_14', 's_15', 's_20', 's_21'
]
TOP_4 = ['s_11', 's_9', 's_4', 's_12']

# 2. LOAD & INITIAL CLEANING
df = pd.read_csv('../data/train_FD001.txt', sep='\s+', header=None, names=COL_NAMES)

# 3. TARGET ENGINEERING (RUL)
# We do this before dropping time_cycles or scaling
max_cycle = df.groupby('unit_nr')['time_cycles'].transform('max')
df['RUL'] = (max_cycle - df['time_cycles']).clip(upper=125) # Combined RUL + Clipping

# 4. FEATURE ENGINEERING (Rolling Stats)
# We do this while data is still grouped by unit_nr
for col in TOP_4:
    group = df.groupby('unit_nr')[col]
    df[f'{col}_roll_mean'] = group.transform(lambda x: x.rolling(10, min_periods=1).mean())
    df[f'{col}_roll_std'] = group.transform(lambda x: x.rolling(10, min_periods=1).std().fillna(0))

# 5. SCALING
# We only scale the columns the model will actually see
scaler = MinMaxScaler()
roll_cols = [f'{c}_roll_mean' for c in TOP_4] + [f'{c}_roll_std' for c in TOP_4]
all_features = FEAT_COLS + roll_cols

df[all_features] = scaler.fit_transform(df[all_features])

# 6. FINAL SELECTION
# No more "honest" or "final" names. Just X and y.
X = df[all_features]
y = df['RUL']

print(f"Preprocessed Data Ready.")
print(f"Features: {X.shape[1]} | Samples: {X.shape[0]}")

Preprocessed Data Ready.
Features: 22 | Samples: 20631


In [8]:
import joblib
joblib.dump(scaler, 'std_scaler.pkl')

['std_scaler.pkl']

In [3]:
# Train-Test Split
# We now split the data using train-test split
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [6]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Train the new model
rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

# Predict using the rolling test set
y_pred = rf.predict(X_test)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"Rolling Model RMSE: {rmse:.2f}")

Rolling Model RMSE: 15.96
